# Gamma-Chain Tensor Networks

This notebook compares two symbolic tensor-network constructions for the Dirac trace.

$\mathrm{Tr}((/\!\!\!p_1 + m I)(/\!\!\!p_2 + m I) \cdots (/\!\!\!p_n + m I)), \qquad /\!\!\!p_i = p_i^\mu \gamma_\mu,$

with cyclic bispinor contractions implementing the trace. Set `has_mass = False` for the massless chain, or `True` to add the same mass symbol `m` to every slashed momentum.

The notebook compares two parallel approaches:

1. The **explicit gamma network**, which keeps the full chain of gamma matrices explicit inside one tensor network.
2. The **pre-contracted gamma network**, which first contracts the gamma chain into Minkowski tensors and then contracts those tensors with the momentum vectors.

In both cases, each momentum four-vector is registered as a sparse symbolic rank-1 tensor. Later cells build evaluators and compiled evaluators for both constructions and compare their numerical performance.

In [1]:
from symbolica import Expression, S, E
from symbolica.community.spenso import (
    ExecutionMode,
    LibraryTensor,
    Representation,
    TensorLibrary,
    TensorNetwork,
    TensorStructure,
)

## Common Helpers

In [2]:
mink = Representation.mink(4)
bis = Representation.bis(4)

def register_sparse_symbolic_vector(lib, name, components):
    tensor = LibraryTensor.sparse(TensorStructure(mink, name=name), Expression)
    for mu, component in enumerate(components):
        tensor[mu] = component
    lib.register(tensor)
    return lib[name]

## Explicit Gamma Network

This construction keeps all `n` gamma matrices explicit in the final tensor network. Each momentum tensor is attached directly to one gamma matrix, and the optional mass term is attached to the same bispinor indices as an identity matrix. The Dirac trace is implemented by cyclic contraction of the bispinor indices.

In [3]:
def explicit_gamma_network(momentum_vectors, mass=None):
    n_vectors = len(momentum_vectors)
    lib = TensorLibrary.hep_lib()
    gamma = lib["spenso::gamma"]
    expr = E("1")
    
    for i, vec in enumerate(momentum_vectors, start=1):

        p_name = S(f"p{i}")
        p = register_sparse_symbolic_vector(lib, p_name, vec)

        mu = S(f"mu{i}")
        left_spinor_index = S(f"s{i}")
        right_spinor_index = S(f"s{1 if i == n_vectors else i + 1}")

        slash_p = p(mu) * gamma(left_spinor_index, right_spinor_index, mu)
        if mass is not None:
            slash_p += mass * bis.id(left_spinor_index, right_spinor_index)

        # The cyclic spinor indices implement the Dirac trace.
        expr *= slash_p

    network = TensorNetwork(expr, lib)

    return (network, lib)

## Pre-contracted Gamma Network

This construction first contracts each cyclic gamma/identity chain into a Minkowski tensor. In the massless case this is the original rank-`n` gamma tensor. With a mass term, the chain is expanded over all choices of momentum-gamma factors and identity-mass factors, and each non-empty gamma subset is registered as its own pre-contracted tensor.

In [4]:
def build_and_register_gamma_trace_tensor(lib, name, n_vectors, gamma_slots=None):
    gamma_slots = set(range(1, n_vectors + 1)) if gamma_slots is None else set(gamma_slots)
    gamma = lib["spenso::gamma"]
    expr = E("1")

    for i in range(1, n_vectors + 1):
        left_spinor_index = S(f"s{i}")
        right_spinor_index = S(f"s{1 if i == n_vectors else i + 1}")

        if i in gamma_slots:
            mu = S(f"mu{i}")
            expr *= gamma(left_spinor_index, right_spinor_index, mu)
        else:
            expr *= bis.id(left_spinor_index, right_spinor_index)

    network = TensorNetwork(expr, lib)
    network.execute(library=lib, mode=ExecutionMode.All)
    gamma_trace_tensor = network.result_tensor(library=lib)
    
    new_gamma_trace_tensor = LibraryTensor.sparse(
        TensorStructure(*([mink] * len(gamma_slots)), name=name),
        Expression,
    )
    structure = gamma_trace_tensor.structure()

    # Sparse tensors raise IndexError on structurally zero entries, so we skip those.
    nonzero_entries = 0
    for flat_index in range(len(gamma_trace_tensor)):
        coordinates = structure[flat_index]
        try:
            value = gamma_trace_tensor[coordinates]
        except IndexError:
            continue
        nonzero_entries += 1
        new_gamma_trace_tensor[coordinates] = value if isinstance(value, Expression) else Expression.num(value)

    lib.register(new_gamma_trace_tensor)
    
    return lib[name]


def pre_contracted_gamma_network(momentum_vectors, mass=None):
    n_vectors = len(momentum_vectors)
    lib = TensorLibrary.hep_lib()

    momenta = []
    for i, vec in enumerate(momentum_vectors, start=1):
        p_name = S(f"p{i}")
        momenta.append(register_sparse_symbolic_vector(lib, p_name, vec))

    if mass is None:
        expr = E("1")
        mu_symbols = []
        for i, p in enumerate(momenta, start=1):
            mu = S(f"mu{i}")
            mu_symbols.append(mu)
            expr *= p(mu)

        gamma_tensor_name = S(f"pre_contracted_gamma_tensor_{n_vectors}")
        gamma_tensor = build_and_register_gamma_trace_tensor(lib, gamma_tensor_name, n_vectors)
        expr *= gamma_tensor(*mu_symbols)
    else:
        expr = E("0")
        for mask in range(1 << n_vectors):
            coefficient = E("1")
            momentum_factor = E("1")
            gamma_slots = []
            mu_symbols = []

            for zero_based_i, p in enumerate(momenta):
                i = zero_based_i + 1
                if mask & (1 << zero_based_i):
                    mu = S(f"mu{i}")
                    gamma_slots.append(i)
                    mu_symbols.append(mu)
                    momentum_factor *= p(mu)
                else:
                    coefficient *= mass

            if not gamma_slots:
                expr += Expression.num(4) * coefficient
                continue

            # Odd gamma traces vanish, and skipping them avoids registering empty tensors.
            if len(gamma_slots) % 2 == 1:
                continue

            mask_name = "".join("p" if mask & (1 << i) else "m" for i in range(n_vectors))
            gamma_tensor_name = S(f"pre_contracted_massive_gamma_tensor_{n_vectors}_{mask_name}")
            gamma_tensor = build_and_register_gamma_trace_tensor(
                lib,
                gamma_tensor_name,
                n_vectors,
                gamma_slots=gamma_slots,
            )
            term_network = TensorNetwork(coefficient * momentum_factor * gamma_tensor(*mu_symbols), lib)
            term_network.execute(library=lib, mode=ExecutionMode.All)
            expr += term_network.result_scalar()

    network = TensorNetwork(expr, lib)
    
    return (network, lib)

## Example Setup

Choose a pool of sample four-vectors, a chain length `n`, and whether the chain includes mass (`has_mass`). The cells below build both the explicit and pre-contracted gamma networks for the same symbolic input, then construct evaluators for both.

Because the pre-contracted construction explicitly materializes one Minkowski tensor per gamma subset when `has_mass` is true, the massive pre-contracted construction grows like a subset expansion on top of the tensor ranks. It is most practical for modest values of `n`.

In [5]:
n_vectors = 6
has_mass = True  # True: Tr((/p1 + m I) ... (/pn + m I)); False: massless chain.
mass = S("m") if has_mass else None

symbolic_momenta = [[S(f"p{i}_{mu}") for mu in range(4)] for i in range(1, n_vectors + 1)]
params = [component for vec in symbolic_momenta for component in vec]
if has_mass:
    params.append(mass)

In [6]:
explicit_network, explicit_lib = explicit_gamma_network(symbolic_momenta, mass=mass)
explicit_network.execute(library=explicit_lib, mode=ExecutionMode.All)
explicit_result_tensor = explicit_network.result_tensor(library=explicit_lib)
explicit_evaluator = explicit_result_tensor.evaluator(
    constants={},
    funs={},
    params=params,
    iterations=50,
    verbose=False,
)
compiled_explicit_evaluator = explicit_evaluator.compile(
    function_name=f"gamma_trace_explicit_{n_vectors}",
    filename=f"gamma_trace_explicit_{n_vectors}.cpp",
    library_name=f"gamma_trace_explicit_{n_vectors}",
    inline_asm="default",
)

In [7]:
pre_contracted_network, pre_contracted_lib = pre_contracted_gamma_network(symbolic_momenta, mass=mass)
pre_contracted_network.execute(library=pre_contracted_lib, mode=ExecutionMode.All)
pre_contracted_result_tensor = pre_contracted_network.result_tensor(library=pre_contracted_lib)
pre_contracted_evaluator = pre_contracted_result_tensor.evaluator(
    constants={},
    funs={},
    params=params,
    iterations=50,
    verbose=False,
)
compiled_pre_contracted_evaluator = pre_contracted_evaluator.compile(
    function_name=f"gamma_trace_pre_contracted_{n_vectors}",
    filename=f"gamma_trace_pre_contracted_{n_vectors}.cpp",
    library_name=f"gamma_trace_pre_contracted_{n_vectors}",
    inline_asm="default",
)

## Timing Comparison

Use the same numerical input batch for both approaches, then compare interpreted and compiled evaluator throughput side by side.

In [8]:
sample_momenta_pool = [
    [1.0, 0.2, -0.1, 0.3],
    [0.7, -0.4, 0.5, 0.1],
    [1.3, 0.0, -0.6, 0.8],
    [0.9, 0.3, 0.2, -0.5],
    [1.1, -0.7, 0.4, 0.6],
    [0.8, 0.5, -0.3, -0.2],
    [1.4, -0.2, 0.1, 0.9],
    [0.6, 0.4, 0.7, -0.1],
    [1.2, -0.5, -0.4, 0.2],
    [0.95, 0.1, 0.6, -0.7],
    [1.05, -0.3, 0.2, 0.4],
    [0.75, 0.6, -0.5, 0.3],
]

sample_momenta = sample_momenta_pool[:n_vectors]
sample_mass = 1.0

sample_dict = {
    symbol: value
    for symbolic_vec, numeric_vec in zip(symbolic_momenta, sample_momenta)
    for symbol, value in zip(symbolic_vec, numeric_vec)
}
if has_mass:
    sample_dict[mass] = sample_mass

sample_input = [component for vec in sample_momenta for component in vec]
if has_mass:
    sample_input.append(sample_mass)

batch_size = 100000
batch_inputs = [sample_input for _ in range(batch_size)]

In [9]:
import time

# start = time.perf_counter()
# explicit_eager = [explicit_network.evaluate(sample_dict, {}) for _ in range(batch_size)]
# explicit_eager_elapsed = time.perf_counter() - start

# start = time.perf_counter()
# pre_contracted_eager = [pre_contracted_network.evaluate(sample_dict, {}) for _ in range(batch_size)]
# pre_contracted_eager_elapsed = time.perf_counter() - start

start = time.perf_counter()
explicit_batch = explicit_evaluator.evaluate_complex(batch_inputs)
explicit_elapsed = time.perf_counter() - start

start = time.perf_counter()
pre_contracted_batch = pre_contracted_evaluator.evaluate_complex(batch_inputs)
pre_contracted_elapsed = time.perf_counter() - start

start = time.perf_counter()
compiled_explicit_batch = compiled_explicit_evaluator.evaluate_complex(batch_inputs)
compiled_explicit_elapsed = time.perf_counter() - start

start = time.perf_counter()
compiled_pre_contracted_batch = compiled_pre_contracted_evaluator.evaluate_complex(batch_inputs)
compiled_pre_contracted_elapsed = time.perf_counter() - start

explicit_value = explicit_batch[0].scalar()
pre_contracted_value = pre_contracted_batch[0].scalar()
compiled_explicit_value = compiled_explicit_batch[0].scalar()
compiled_pre_contracted_value = compiled_pre_contracted_batch[0].scalar()

print(f"Number of momenta: {n_vectors}")
print(f"Batch size: {batch_size}")

print(f"\nExplicit gamma network:")
print(f"    Value: {explicit_value}")
print(f"    Compiled value: {compiled_explicit_value}")
# print(f"    Eager time: {1.e6 * explicit_eager_elapsed / batch_size:.3f} µs/pt")
print(f"    Evaluator time: {1.e6 * explicit_elapsed / batch_size:.3f} µs/pt")
print(f"    Compiled time: {1.e6 * compiled_explicit_elapsed / batch_size:.3f} µs/pt")

print(f"\nPre-contracted gamma network:")
print(f"    Value: {pre_contracted_value}")
print(f"    Compiled value: {compiled_pre_contracted_value}")
# print(f"    Eager time: {1.e6 * pre_contracted_eager_elapsed / batch_size:.3f} µs/pt")
print(f"    Evaluator time: {1.e6 * pre_contracted_elapsed / batch_size:.3f} µs/pt")
print(f"    Compiled time: {1.e6 * compiled_pre_contracted_elapsed / batch_size:.3f} µs/pt")

Number of momenta: 6
Batch size: 100000

Explicit gamma network:
    Value: 165.088364000000
    Compiled value: 165.088364000000+2.22044604925031e-16𝑖
    Evaluator time: 1.077 µs/pt
    Compiled time: 0.508 µs/pt

Pre-contracted gamma network:
    Value: 165.088364000000
    Compiled value: 165.088364000000
    Evaluator time: 1.803 µs/pt
    Compiled time: 0.864 µs/pt
